In [ ]:
# Import pandas for data handling
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Import plotly express for interactive charts
import plotly.express as px

# Import plotly graph objects for custom ROC curve plots
import plotly.graph_objects as go

# Import train_test_split for splitting data into train and test sets
from sklearn.model_selection import train_test_split

# Import StandardScaler to normalise feature values
from sklearn.preprocessing import StandardScaler

# Import Pipeline to chain preprocessing and model steps cleanly
from sklearn.pipeline import Pipeline

# Import Gaussian Naive Bayes for base learner 1
from sklearn.naive_bayes import GaussianNB

# Import Logistic Regression for base learner 2
from sklearn.linear_model import LogisticRegression

# Import VotingClassifier to combine base learners into an ensemble
from sklearn.ensemble import VotingClassifier

# Import Decision Tree Regressor for regression modelling
from sklearn.tree import DecisionTreeRegressor, plot_tree

# Import classification evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Import regression evaluation metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Import matplotlib for decision tree visualisation
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("All libraries imported successfully!")


All libraries imported successfully!


#part A - Ensemble Classification:Voting Classifier

In [ ]:
# Load the cleaned classification dataset saved from Notebook 1
df_cls = pd.read_csv('/content/drive/MyDrive/CW_ML_20240856/dataset1_classification_cleaned.csv')

# Drop any remaining missing values
df_cls = df_cls.dropna()

# Print dataset shape
print("Classification Dataset Shape:", df_cls.shape)
print("\nColumns:", df_cls.columns.tolist())
df_cls.head()


Classification Dataset Shape: (58628, 17)

Columns: ['age', 'income', 'employment_length', 'loan_amount', 'loan_interest_rate', 'loan_income_ratio', 'payment_default_on_file', 'credit_history_length', 'loan_approval_status', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE']


,age,income,employment_length,loan_amount,loan_interest_rate,loan_income_ratio,payment_default_on_file,credit_history_length,loan_approval_status,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,21.0,12000,0.0,15000,6.99,0.12,0,4.0,0,0,1,0,1,0,0,0,0
1,21.0,13200,2.0,22500,16.77,0.19,1,3.0,0,0,1,0,1,0,0,0,0
2,23.0,9600,5.0,22500,12.42,0.31,0,3.0,0,0,0,1,0,0,1,0,0
3,40.0,126000,3.0,22500,8.00,0.19,0,11.0,0,0,0,1,1,0,0,0,0
4,40.0,90000,3.0,22500,12.42,0.39,0,14.0,0,0,0,0,0,1,0,0,0


#Define Features and Target

In [ ]:
# Separate input features (X) from target variable (y)
X_cls = df_cls.drop(columns=['loan_approval_status'])

# Assign target variable
y_cls = df_cls['loan_approval_status']

# Remove rows with any NaN values in X
mask = X_cls.notna().all(axis=1)
X_cls = X_cls[mask].reset_index(drop=True)
y_cls = y_cls[mask].reset_index(drop=True)

print("Feature shape:", X_cls.shape)
print("Target shape:", y_cls.shape)
print("\nFeature names:", X_cls.columns.tolist())


Feature shape: (58628, 16)
Target shape: (58628,)

Feature names: ['age', 'income', 'employment_length', 'loan_amount', 'loan_interest_rate', 'loan_income_ratio', 'payment_default_on_file', 'credit_history_length', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE']


#Train-test Split(80/20)

In [ ]:
# Split dataset into 80% training and 20% test
# random_state=42 ensures all models use identical test instances
# stratify=y_cls maintains class ratio in both subsets
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls,
    test_size=0.20,
    random_state=42,
    stratify=y_cls
)

print("Training set size:", X_train_c.shape)
print("Test set size    :", X_test_c.shape)
print("\nClass ratio – Training set:")
print(y_train_c.value_counts(normalize=True).round(3))
print("\nClass ratio – Test set:")
print(y_test_c.value_counts(normalize=True).round(3))


Training set size: (46902, 16)
Test set size    : (11726, 16)

Class ratio – Training set:
loan_approval_status
0    0.858
1    0.142
Name: proportion, dtype: float64

Class ratio – Test set:
loan_approval_status
0    0.858
1    0.142
Name: proportion, dtype: float64


# Task 5(f)(i) - Build voting Ensemble Classifier

In [ ]:
# Import Pipeline to wrap preprocessing and model for each base learner
from sklearn.pipeline import Pipeline

# Define Base Learner 1: Naive Bayes wrapped in a Pipeline
# NB does not need scaling — FunctionTransformer passes data through unchanged
from sklearn.preprocessing import FunctionTransformer
nb_pipeline = Pipeline([
    ('passthrough', FunctionTransformer()),   # no scaling needed for NB
    ('nb', GaussianNB())                      # Gaussian Naive Bayes classifier
])

# Define Base Learner 2: Logistic Regression wrapped in a Pipeline with scaling
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),             # scale features before LR
    ('lr', LogisticRegression(max_iter=1000, random_state=42))  # LR classifier
])

# Combine NB and LR into a soft Voting Ensemble Classifier
# voting='soft' averages predicted probabilities from both base learners
voting_clf = VotingClassifier(
    estimators=[
        ('nb', nb_pipeline),    # Base Learner 1: Naive Bayes
        ('lr', lr_pipeline)     # Base Learner 2: Logistic Regression
    ],
    voting='soft'
)

# Fit the ensemble classifier on the raw training data
# Each pipeline handles its own preprocessing internally
voting_clf.fit(X_train_c, y_train_c)

print("Voting Ensemble Classifier (NB + LR) trained successfully.")


Voting Ensemble Classifier (NB + LR) trained successfully.


# Task 5(f)(ii) - Evaluate base Learner 1:Naive bayes

In [ ]:
# Train and evaluate Naive Bayes individually for comparison
# NB does not need scaling — use raw training and test data
nb_base = GaussianNB()
nb_base.fit(X_train_c, y_train_c)

# Predict class labels on test data
nb_pred = nb_base.predict(X_test_c)

# Get predicted probabilities for class 1 for AUC-ROC
nb_prob = nb_base.predict_proba(X_test_c)[:, 1]

# Print all evaluation metrics for NB base learner
print("="*55)
print("  Base Learner 1 – Naive Bayes (NB)")
print("="*55)
print(f"  Accuracy  : {accuracy_score(y_test_c, nb_pred):.4f}")
print(f"  Precision : {precision_score(y_test_c, nb_pred, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(y_test_c, nb_pred, zero_division=0):.4f}")
print(f"  F1-Score  : {f1_score(y_test_c, nb_pred, zero_division=0):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_test_c, nb_prob):.4f}")
print("\nClassification Report – NB:")
print(classification_report(y_test_c, nb_pred))


  Base Learner 1 – Naive Bayes (NB)
  Accuracy  : 0.8865
  Precision : 0.7493
  Recall    : 0.3044
  F1-Score  : 0.4329
  ROC-AUC   : 0.8440

Classification Report – NB:
              precision    recall  f1-score   support

           0       0.89      0.98      0.94     10057
           1       0.75      0.30      0.43      1669

    accuracy                           0.89     11726
   macro avg       0.82      0.64      0.68     11726
weighted avg       0.87      0.89      0.87     11726



In [ ]:
# Plot confusion matrix for Naive Bayes base learner
cm_nb = confusion_matrix(y_test_c, nb_pred)
fig_nb = px.imshow(
    cm_nb, text_auto=True,
    color_continuous_scale='Blues',
    labels=dict(x='Predicted Label', y='Actual Label'),
    x=['Rejected (0)', 'Approved (1)'],
    y=['Rejected (0)', 'Approved (1)'],
    title='Confusion Matrix – Base Learner 1: Naive Bayes (NB)'
)
# Centre the title
fig_nb.update_layout(title_x=0.5)
fig_nb.show()


#Task 5(F)(ii) Evaluate Base Learner 2: Logistic Regression(LR)

In [ ]:
# Train and evaluate Logistic Regression individually for comparison
# LR requires StandardScaler — fit scaler on training data only
scaler_lr = StandardScaler()
X_train_lr = scaler_lr.fit_transform(X_train_c)
X_test_lr  = scaler_lr.transform(X_test_c)

# Train Logistic Regression on scaled training data
lr_base = LogisticRegression(max_iter=1000, random_state=42)
lr_base.fit(X_train_lr, y_train_c)

# Predict class labels on scaled test data
lr_pred = lr_base.predict(X_test_lr)

# Get predicted probabilities for class 1 for AUC-ROC
lr_prob = lr_base.predict_proba(X_test_lr)[:, 1]

# Print all evaluation metrics for LR base learner
print("="*55)
print("  Base Learner 2 – Logistic Regression (LR)")
print("="*55)
print(f"  Accuracy  : {accuracy_score(y_test_c, lr_pred):.4f}")
print(f"  Precision : {precision_score(y_test_c, lr_pred, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(y_test_c, lr_pred, zero_division=0):.4f}")
print(f"  F1-Score  : {f1_score(y_test_c, lr_pred, zero_division=0):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_test_c, lr_prob):.4f}")
print("\nClassification Report – LR:")
print(classification_report(y_test_c, lr_pred))


  Base Learner 2 – Logistic Regression (LR)
  Accuracy  : 0.8941
  Precision : 0.7163
  Recall    : 0.4236
  F1-Score  : 0.5324
  ROC-AUC   : 0.8864

Classification Report – LR:
              precision    recall  f1-score   support

           0       0.91      0.97      0.94     10057
           1       0.72      0.42      0.53      1669

    accuracy                           0.89     11726
   macro avg       0.81      0.70      0.74     11726
weighted avg       0.88      0.89      0.88     11726



In [ ]:
# Plot confusion matrix for Logistic Regression base learner
cm_lr = confusion_matrix(y_test_c, lr_pred)
fig_lr = px.imshow(
    cm_lr, text_auto=True,
    color_continuous_scale='Greens',
    labels=dict(x='Predicted Label', y='Actual Label'),
    x=['Rejected (0)', 'Approved (1)'],
    y=['Rejected (0)', 'Approved (1)'],
    title='Confusion Matrix – Base Learner 2: Logistic Regression (LR)'
)
# Centre the title
fig_lr.update_layout(title_x=0.5)
fig_lr.show()


# Task 5(f)(ii) = Classification Report: Base Learner 1(NB)

In [ ]:
# Task 5(f)(ii) — Classification Report: Base Learner 1 – Naive Bayes (NB)
print('Classification Report – Base Learner 1: Naive Bayes (NB)')
print('='*55)
print(classification_report(
    y_test_c, nb_pred,
    target_names=['Rejected (0)', 'Approved (1)'],
    zero_division=0
))

Classification Report – Base Learner 1: Naive Bayes (NB)
              precision    recall  f1-score   support

Rejected (0)       0.89      0.98      0.94     10057
Approved (1)       0.75      0.30      0.43      1669

    accuracy                           0.89     11726
   macro avg       0.82      0.64      0.68     11726
weighted avg       0.87      0.89      0.87     11726



#AUc - ROc curve: Base Learner1 (NB)

In [ ]:
# Task 5(f)(ii) — AUC-ROC Curve: Base Learner 1 – Naive Bayes (NB)
fpr_nb, tpr_nb, _ = roc_curve(y_test_c, nb_prob)
auc_nb = roc_auc_score(y_test_c, nb_prob)

fig_roc_nb = go.Figure()
fig_roc_nb.add_trace(go.Scatter(
    x=fpr_nb, y=tpr_nb, mode='lines',
    name=f'Naive Bayes (AUC={auc_nb:.3f})',
    line=dict(color='#636EFA', width=2)
))
fig_roc_nb.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    name='Random Classifier',
    line=dict(color='grey', dash='dash', width=1)
))
fig_roc_nb.update_layout(
    title='AUC-ROC Curve – Base Learner 1: Naive Bayes (NB)',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    title_x=0.5, width=700, height=480
)
fig_roc_nb.show()
print(f'Naive Bayes AUC-ROC: {auc_nb:.4f}')

Naive Bayes AUC-ROC: 0.8440


#Task 5(F)(ii) Evaluate Base Learner 2: Logistic Regression(LR)

In [ ]:
# Train and evaluate Logistic Regression individually for comparison
# LR requires StandardScaler — fit scaler on training data only
scaler_lr = StandardScaler()
X_train_lr = scaler_lr.fit_transform(X_train_c)
X_test_lr  = scaler_lr.transform(X_test_c)

# Train Logistic Regression on scaled training data
lr_base = LogisticRegression(max_iter=1000, random_state=42)
lr_base.fit(X_train_lr, y_train_c)

# Predict class labels on scaled test data
lr_pred = lr_base.predict(X_test_lr)

# Get predicted probabilities for class 1 for AUC-ROC
lr_prob = lr_base.predict_proba(X_test_lr)[:, 1]

# Print all evaluation metrics for LR base learner
print("="*55)
print("  Base Learner 2 – Logistic Regression (LR)")
print("="*55)
print(f"  Accuracy  : {accuracy_score(y_test_c, lr_pred):.4f}")
print(f"  Precision : {precision_score(y_test_c, lr_pred, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(y_test_c, lr_pred, zero_division=0):.4f}")
print(f"  F1-Score  : {f1_score(y_test_c, lr_pred, zero_division=0):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_test_c, lr_prob):.4f}")
print("\nClassification Report – LR:")
print(classification_report(y_test_c, lr_pred))


  Base Learner 2 – Logistic Regression (LR)
  Accuracy  : 0.8941
  Precision : 0.7163
  Recall    : 0.4236
  F1-Score  : 0.5324
  ROC-AUC   : 0.8864

Classification Report – LR:
              precision    recall  f1-score   support

           0       0.91      0.97      0.94     10057
           1       0.72      0.42      0.53      1669

    accuracy                           0.89     11726
   macro avg       0.81      0.70      0.74     11726
weighted avg       0.88      0.89      0.88     11726



In [ ]:
# Plot confusion matrix for Logistic Regression base learner
cm_lr = confusion_matrix(y_test_c, lr_pred)
fig_lr = px.imshow(
    cm_lr, text_auto=True,
    color_continuous_scale='Greens',
    labels=dict(x='Predicted Label', y='Actual Label'),
    x=['Rejected (0)', 'Approved (1)'],
    y=['Rejected (0)', 'Approved (1)'],
    title='Confusion Matrix – Base Learner 2: Logistic Regression (LR)'
)
# Centre the title
fig_lr.update_layout(title_x=0.5)
fig_lr.show()


# Task 5(f)(ii) — AUC-ROC Curve: Base Learner 2 (LR)

In [ ]:
# Task 5(f)(ii) — AUC-ROC Curve: Base Learner 2 – Logistic Regression (LR)
fpr_lr, tpr_lr, _ = roc_curve(y_test_c, lr_prob)
auc_lr = roc_auc_score(y_test_c, lr_prob)

fig_roc_lr = go.Figure()
fig_roc_lr.add_trace(go.Scatter(
    x=fpr_lr, y=tpr_lr, mode='lines',
    name=f'Logistic Regression (AUC={auc_lr:.3f})',
    line=dict(color='#EF553B', width=2)
))
fig_roc_lr.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    name='Random Classifier',
    line=dict(color='grey', dash='dash', width=1)
))
fig_roc_lr.update_layout(
    title='AUC-ROC Curve – Base Learner 2: Logistic Regression (LR)',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    title_x=0.5, width=700, height=480
)
fig_roc_lr.show()
print(f'Logistic Regression AUC-ROC: {auc_lr:.4f}')

Logistic Regression AUC-ROC: 0.8864


# task 5(f)(ii) - Evaluate voting Ensemble classifier

In [ ]:
# Predict class labels using the trained Voting Ensemble Classifier
vc_pred = voting_clf.predict(X_test_c)

# Get predicted probabilities from the ensemble for AUC-ROC
vc_prob = voting_clf.predict_proba(X_test_c)[:, 1]

# Calculate all classification metrics for the ensemble
vc_acc  = accuracy_score(y_test_c, vc_pred)
vc_prec = precision_score(y_test_c, vc_pred, zero_division=0)
vc_rec  = recall_score(y_test_c, vc_pred, zero_division=0)
vc_f1   = f1_score(y_test_c, vc_pred, zero_division=0)
vc_auc  = roc_auc_score(y_test_c, vc_prob)

# Print all metrics for the ensemble
print("="*55)
print("  Voting Ensemble Classifier (NB + LR) – Test Results")
print("="*55)
print(f"  Accuracy  : {vc_acc:.4f}")
print(f"  Precision : {vc_prec:.4f}")
print(f"  Recall    : {vc_rec:.4f}")
print(f"  F1-Score  : {vc_f1:.4f}")
print(f"  ROC-AUC   : {vc_auc:.4f}")
print("="*55)
print("\nClassification Report – Voting Ensemble:")
print(classification_report(y_test_c, vc_pred))


  Voting Ensemble Classifier (NB + LR) – Test Results
  Accuracy  : 0.8934
  Precision : 0.7622
  Recall    : 0.3649
  F1-Score  : 0.4935
  ROC-AUC   : 0.8792

Classification Report – Voting Ensemble:
              precision    recall  f1-score   support

           0       0.90      0.98      0.94     10057
           1       0.76      0.36      0.49      1669

    accuracy                           0.89     11726
   macro avg       0.83      0.67      0.72     11726
weighted avg       0.88      0.89      0.88     11726



In [ ]:
# Plot confusion matrix for the Voting Ensemble Classifier
cm_vc = confusion_matrix(y_test_c, vc_pred)
fig_vc = px.imshow(
    cm_vc, text_auto=True,
    color_continuous_scale='Purples',
    labels=dict(x='Predicted Label', y='Actual Label'),
    x=['Rejected (0)', 'Approved (1)'],
    y=['Rejected (0)', 'Approved (1)'],
    title='Confusion Matrix – Voting Ensemble Classifier (NB + LR)'
)
# Centre the title
fig_vc.update_layout(title_x=0.5)
fig_vc.show()


# Task 5(f)(ii) — Classification Report: Voting Ensemble (NB + LR)

In [ ]:
# Task 5(f)(ii) — Classification Report: Voting Ensemble Classifier (NB + LR)
print('Classification Report – Voting Ensemble Classifier (NB + LR)')
print('='*55)
print(classification_report(
    y_test_c, vc_pred,
    target_names=['Rejected (0)', 'Approved (1)'],
    zero_division=0
))

Classification Report – Voting Ensemble Classifier (NB + LR)
              precision    recall  f1-score   support

Rejected (0)       0.90      0.98      0.94     10057
Approved (1)       0.76      0.36      0.49      1669

    accuracy                           0.89     11726
   macro avg       0.83      0.67      0.72     11726
weighted avg       0.88      0.89      0.88     11726



# Task 5(f)(ii) — AUC-ROC Curve: Voting Ensemble (NB + LR)

In [ ]:
# Task 5(f)(ii) — AUC-ROC Curve: Voting Ensemble Classifier (NB + LR)
fpr_vc, tpr_vc, _ = roc_curve(y_test_c, vc_prob)
auc_vc = roc_auc_score(y_test_c, vc_prob)

fig_roc_vc = go.Figure()
fig_roc_vc.add_trace(go.Scatter(
    x=fpr_vc, y=tpr_vc, mode='lines',
    name=f'Voting Ensemble NB+LR (AUC={auc_vc:.3f})',
    line=dict(color='#00CC96', width=2)
))
fig_roc_vc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    name='Random Classifier',
    line=dict(color='grey', dash='dash', width=1)
))
fig_roc_vc.update_layout(
    title='AUC-ROC Curve – Voting Ensemble Classifier (NB + LR)',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    title_x=0.5, width=700, height=480
)
fig_roc_vc.show()
print(f'Voting Ensemble AUC-ROC: {auc_vc:.4f}')

Voting Ensemble AUC-ROC: 0.8792


# Auc-ROC Curves - NB vs LR vs Voting Ensemble

In [ ]:
# Plot AUC-ROC curves for both base learners and the ensemble on one graph
fig_roc = go.Figure()

# Define all three models with their probabilities and display colours
models_roc = {
    'Naive Bayes (NB)': (nb_prob, '#636EFA'),
    'Logistic Regression (LR)': (lr_prob, '#EF553B'),
    'Voting Ensemble (NB+LR)': (vc_prob, '#00CC96')
}

# Loop through each model and add its ROC curve
for name, (prob, color) in models_roc.items():
    # Calculate false positive rate and true positive rate
    fpr, tpr, _ = roc_curve(y_test_c, prob)
    # Calculate AUC score
    auc_val = roc_auc_score(y_test_c, prob)
    # Add trace to figure
    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines',
        name=f'{name} (AUC={auc_val:.3f})',
        line=dict(color=color, width=2)
    ))

# Add random classifier baseline diagonal line
fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    name='Random Classifier',
    line=dict(color='grey', dash='dash', width=1)
))

# Update chart layout
fig_roc.update_layout(
    title='AUC-ROC Curves: NB vs LR vs Voting Ensemble',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    legend=dict(x=0.55, y=0.1),
    title_x=0.5, width=700, height=480
)
fig_roc.show()


# Part B - Regression Decision Trees:Predicting Maximum Loan Amount

In [ ]:
# Load the cleaned regression dataset saved from Notebook 1
# This dataset contains approved loans only with max_allowed_loan as target
df_reg = pd.read_csv('/content/drive/MyDrive/CW_ML_20240856/dataset2_regression_cleaned.csv')

# Drop any remaining rows with missing values
df_reg = df_reg.dropna()

# Print dataset shape and column names to confirm correct load
print("Regression Dataset Shape:", df_reg.shape)
print("\nColumns:", df_reg.columns.tolist())
df_reg.head()


Regression Dataset Shape: (50278, 17)

Columns: ['age', 'income', 'employment_length', 'loan_amount', 'loan_interest_rate', 'loan_income_ratio', 'payment_default_on_file', 'credit_history_length', 'max_allowed_loan', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE']


,age,income,employment_length,loan_amount,loan_interest_rate,loan_income_ratio,payment_default_on_file,credit_history_length,max_allowed_loan,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,40.0,132500,3.0,22500,8.00,0.19,0,11.0,35000.0,0,0,1,1,0,0,0,0
1,40.0,90000,3.0,22500,12.42,0.38,0,14.0,35000.0,0,0,0,0,1,0,0,0
2,40.0,131004,9.0,22500,7.90,0.23,0,13.0,30000.0,0,0,0,0,0,0,0,1
3,40.0,132500,9.0,22500,11.89,0.17,0,11.0,25000.0,0,0,0,0,0,0,0,0
4,40.0,132000,3.0,22500,16.82,0.22,0,15.5,25000.0,0,0,0,0,1,0,0,0


#Define Feature (X) and target(y) for regression

In [ ]:
# Drop the target column to create the input feature matrix X
X_reg = df_reg.drop(columns=['max_allowed_loan'])

# Assign the target column to y
y_reg = df_reg['max_allowed_loan']

# Print feature shape and target shape
print("Feature shape:", X_reg.shape)
print("Target shape :", y_reg.shape)

# Print feature names — screenshot this for Case Study B Task 1
print("\nRetained feature names:")
print(X_reg.columns.tolist())

# Print target variable summary statistics
print("\nTarget Variable – Maximum Loan Amount:")
print(f"  Mean   : £{y_reg.mean():,.2f}")
print(f"  Median : £{y_reg.median():,.2f}")
print(f"  Std    : £{y_reg.std():,.2f}")
print(f"  Min    : £{y_reg.min():,.2f}")
print(f"  Max    : £{y_reg.max():,.2f}")


Feature shape: (50278, 16)
Target shape : (50278,)

Retained feature names:
['age', 'income', 'employment_length', 'loan_amount', 'loan_interest_rate', 'loan_income_ratio', 'payment_default_on_file', 'credit_history_length', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE']

Target Variable – Maximum Loan Amount:
  Mean   : £78,280.90
  Median : £69,418.50
  Std    : £39,397.47
  Min    : £232.00
  Max    : £173,811.75


feature Scaling

In [ ]:
# Apply StandardScaler – fit on training data only, transform both sets
scaler_c = StandardScaler()
X_train_cs = scaler_c.fit_transform(X_train_c)
X_test_cs  = scaler_c.transform(X_test_c)

print("StandardScaler applied successfully.")


StandardScaler applied successfully.


#Task 6a - Train base Learner 1:Naive Bayes(NB0


In [ ]:
# Train Naive Bayes base learner using unscaled features
nb_base = GaussianNB()
nb_base.fit(X_train_c, y_train_c)

# Predict and evaluate NB base learner
nb_pred  = nb_base.predict(X_test_c)
nb_prob  = nb_base.predict_proba(X_test_c)[:, 1]

print("Base Learner 1 – Naive Bayes (NB) Results:")
print(f"  Accuracy  : {accuracy_score(y_test_c, nb_pred):.4f}")
print(f"  Precision : {precision_score(y_test_c, nb_pred, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(y_test_c, nb_pred, zero_division=0):.4f}")
print(f"  F1-Score  : {f1_score(y_test_c, nb_pred, zero_division=0):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_test_c, nb_prob):.4f}")


Base Learner 1 – Naive Bayes (NB) Results:
  Accuracy  : 0.8865
  Precision : 0.7493
  Recall    : 0.3044
  F1-Score  : 0.4329
  ROC-AUC   : 0.8440


#Task 6a – Train Base Learner 2: Logistic Regression (LR)

In [ ]:
# Train Logistic Regression base learner using scaled features
lr_base = LogisticRegression(max_iter=1000, random_state=42)

# Fit LR on scaled training data
lr_base.fit(X_train_cs, y_train_c)

# Predict on scaled test data
lr_pred = lr_base.predict(X_test_cs)

# Get predicted probabilities for AUC-ROC calculation
lr_prob = lr_base.predict_proba(X_test_cs)[:, 1]

print("Base Learner 2 – Logistic Regression (LR) Results:")
print(f"  Accuracy  : {accuracy_score(y_test_c, lr_pred):.4f}")
print(f"  Precision : {precision_score(y_test_c, lr_pred, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(y_test_c, lr_pred, zero_division=0):.4f}")
print(f"  F1-Score  : {f1_score(y_test_c, lr_pred, zero_division=0):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_test_c, lr_prob):.4f}")


Base Learner 2 – Logistic Regression (LR) Results:
  Accuracy  : 0.8954
  Precision : 0.7264
  Recall    : 0.4248
  F1-Score  : 0.5361
  ROC-AUC   : 0.8864


#Task 6b – Build Voting Ensemble Classifier (NB + LR)

In [ ]:
# Combine NB and LR into a soft Voting Ensemble Classifier
# Soft voting averages predicted probabilities from both base learners
voting_clf = VotingClassifier(
    estimators=[
        ('nb', GaussianNB()),
        ('lr', LogisticRegression(max_iter=1000, random_state=42))
    ],
    voting='soft'
)

# Fit ensemble on training data (scaled, because LR needs scaling;
# VotingClassifier applies the same data to both learners here)
voting_clf.fit(X_train_cs, y_train_c)

# Predict on test set
vc_pred = voting_clf.predict(X_test_cs)
vc_prob = voting_clf.predict_proba(X_test_cs)[:, 1]

print("Voting Ensemble Classifier trained successfully.")


Voting Ensemble Classifier trained successfully.


# Task 6c – Evaluate Voting Ensemble Classifier

In [ ]:
# Calculate all classification evaluation metrics for the ensemble
vc_acc  = accuracy_score(y_test_c, vc_pred)
vc_prec = precision_score(y_test_c, vc_pred, zero_division=0)
vc_rec  = recall_score(y_test_c, vc_pred, zero_division=0)
vc_f1   = f1_score(y_test_c, vc_pred, zero_division=0)
vc_auc  = roc_auc_score(y_test_c, vc_prob)

print("="*55)
print("  Voting Ensemble (NB + LR) – Test Results")
print("="*55)
print(f"  Accuracy  : {vc_acc:.4f}")
print(f"  Precision : {vc_prec:.4f}")
print(f"  Recall    : {vc_rec:.4f}")
print(f"  F1-Score  : {vc_f1:.4f}")
print(f"  ROC-AUC   : {vc_auc:.4f}")
print("="*55)
print("\nClassification Report:")
print(classification_report(y_test_c, vc_pred))


  Voting Ensemble (NB + LR) – Test Results
  Accuracy  : 0.8627
  Precision : 0.5140
  Recall    : 0.6483
  F1-Score  : 0.5734
  ROC-AUC   : 0.8828

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.90      0.92     10057
           1       0.51      0.65      0.57      1669

    accuracy                           0.86     11726
   macro avg       0.73      0.77      0.75     11726
weighted avg       0.88      0.86      0.87     11726



#confusion Matrix-Voting Ensemble

In [ ]:
# Build confusion matrix for voting ensemble classifier
cm_vc = confusion_matrix(y_test_c, vc_pred)

# Plot confusion matrix as heatmap
fig_cm = px.imshow(
    cm_vc,
    text_auto=True,
    color_continuous_scale='Purples',
    labels=dict(x='Predicted Label', y='Actual Label'),
    x=['Rejected (0)', 'Approved (1)'],
    y=['Rejected (0)', 'Approved (1)'],
    title='Confusion Matrix – Voting Ensemble (NB + LR)'
)
fig_cm.update_layout(title_x=0.5, title_font_size=15)
fig_cm.show()


#AUC-ROC Curve- Ensemble vs Base Learners

In [ ]:
# Plot ROC curves for all three models: NB, LR, and Ensemble
fig_roc = go.Figure()

# Define models, probabilities, and display colours
models_roc = {
    'Naive Bayes (NB)': (nb_prob, '#636EFA'),
    'Logistic Regression (LR)': (lr_prob, '#EF553B'),
    'Voting Ensemble (NB+LR)': (vc_prob, '#00CC96')
}

# Plot each model's ROC curve
for name, (prob, color) in models_roc.items():
    fpr, tpr, _ = roc_curve(y_test_c, prob)
    auc_val = roc_auc_score(y_test_c, prob)
    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines',
        name=f'{name} (AUC={auc_val:.3f})',
        line=dict(color=color, width=2)
    ))

# Add random classifier baseline
fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    name='Random Classifier',
    line=dict(color='grey', dash='dash', width=1)
))

fig_roc.update_layout(
    title='AUC-ROC Curves: NB vs LR vs Voting Ensemble',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    legend=dict(x=0.55, y=0.1),
    title_x=0.5, width=700, height=480
)
fig_roc.show()


#Summary Table

In [ ]:
# Collect all evaluation results into a summary DataFrame
summary_cls = pd.DataFrame([
    {'Model': 'Naive Bayes (NB)',
     'Accuracy': accuracy_score(y_test_c, nb_pred),
     'Precision': precision_score(y_test_c, nb_pred, zero_division=0),
     'Recall': recall_score(y_test_c, nb_pred, zero_division=0),
     'F1-Score': f1_score(y_test_c, nb_pred, zero_division=0),
     'ROC-AUC': roc_auc_score(y_test_c, nb_prob)},
    {'Model': 'Logistic Regression (LR)',
     'Accuracy': accuracy_score(y_test_c, lr_pred),
     'Precision': precision_score(y_test_c, lr_pred, zero_division=0),
     'Recall': recall_score(y_test_c, lr_pred, zero_division=0),
     'F1-Score': f1_score(y_test_c, lr_pred, zero_division=0),
     'ROC-AUC': roc_auc_score(y_test_c, lr_prob)},
    {'Model': 'Voting Ensemble (NB+LR)',
     'Accuracy': vc_acc, 'Precision': vc_prec,
     'Recall': vc_rec, 'F1-Score': vc_f1, 'ROC-AUC': vc_auc}
]).set_index('Model')

print("Classification Performance Summary (Test Set):")
print(summary_cls.round(4).to_string())


Classification Performance Summary (Test Set):
                          Accuracy  Precision  Recall  F1-Score  ROC-AUC
Model                                                                   
Naive Bayes (NB)            0.8790     0.6796  0.2834    0.4000   0.8305
Logistic Regression (LR)    0.8954     0.7264  0.4248    0.5361   0.8864
Voting Ensemble (NB+LR)     0.8627     0.5140  0.6483    0.5734   0.8828


#Part B - Regression Decision trees:Pedicting Maximum Loan Amount

In [ ]:
# Load the cleaned regression dataset saved from Notebook 1
df_reg = pd.read_csv('/content/drive/MyDrive/CW_ML_20240856/dataset2_regression_cleaned.csv')

# Drop any remaining missing values
df_reg = df_reg.dropna()

print("Regression Dataset Shape:", df_reg.shape)
print("\nColumns:", df_reg.columns.tolist())
df_reg.head()


Regression Dataset Shape: (50278, 17)

Columns: ['age', 'income', 'employment_length', 'loan_amount', 'loan_interest_rate', 'loan_income_ratio', 'payment_default_on_file', 'credit_history_length', 'max_allowed_loan', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE']


,age,income,employment_length,loan_amount,loan_interest_rate,loan_income_ratio,payment_default_on_file,credit_history_length,max_allowed_loan,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,40.0,132500,3.0,22500,8.00,0.19,0,11.0,35000.0,0,0,1,1,0,0,0,0
1,40.0,90000,3.0,22500,12.42,0.38,0,14.0,35000.0,0,0,0,0,1,0,0,0
2,40.0,131004,9.0,22500,7.90,0.23,0,13.0,30000.0,0,0,0,0,0,0,0,1
3,40.0,132500,9.0,22500,11.89,0.17,0,11.0,25000.0,0,0,0,0,0,0,0,0
4,40.0,132000,3.0,22500,16.82,0.22,0,15.5,25000.0,0,0,0,0,1,0,0,0


Define Features and Target for Regression

In [ ]:
# Separate input features (X) from regression target variable
X_reg = df_reg.drop(columns=['max_allowed_loan'])

# Assign target variable
y_reg = df_reg['max_allowed_loan']

print("Feature shape:", X_reg.shape)
print("Target shape :", y_reg.shape)
print("\nTarget statistics:")
print(f"  Mean   : £{y_reg.mean():,.2f}")
print(f"  Median : £{y_reg.median():,.2f}")
print(f"  Std    : £{y_reg.std():,.2f}")
print(f"  Min    : £{y_reg.min():,.2f}")
print(f"  Max    : £{y_reg.max():,.2f}")


Feature shape: (50278, 16)
Target shape : (50278,)

Target statistics:
  Mean   : £78,280.90
  Median : £69,418.50
  Std    : £39,397.47
  Min    : £232.00
  Max    : £173,811.75


#train-test Split(80/20)

In [ ]:
# Split regression data 80% training, 20% test
# random_state=42 ensures reproducibility
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg,
    test_size=0.20,
    random_state=42
)

print("Training set size:", X_train_r.shape)
print("Test set size    :", X_test_r.shape)


Training set size: (40222, 16)
Test set size    : (10056, 16)


#Task 2b(i)-DT-1: FUlly Grown Decision Tree Regressor

In [ ]:
# Import Decision Tree Regressor
from sklearn.tree import DecisionTreeRegressor

# Declare DT-1 as a fully grown Decision Tree Regressor (no depth limit)
dt1 = DecisionTreeRegressor(random_state=42)

# Fit DT-1 on the training data
dt1.fit(X_train_r, y_train_r)

print("DT-1 (Fully Grown) trained successfully.")
print(f"  Tree depth   : {dt1.get_depth()}")
print(f"  Leaf nodes   : {dt1.get_n_leaves()}")


DT-1 (Fully Grown) trained successfully.
  Tree depth   : 30
  Leaf nodes   : 21673


#Task 2b(i) - DT-2:Pruned Decision tree Regressor(max_depth = 4)

In [ ]:
# Declare DT-2 as a pruned Decision Tree Regressor limited to 4 levels
# Pre-pruning via max_depth controls tree complexity
dt2 = DecisionTreeRegressor(max_depth=4, random_state=42)

# Fit DT-2 on the training data
dt2.fit(X_train_r, y_train_r)

print("DT-2 (Pruned, max_depth=4) trained successfully.")
print(f"  Tree depth   : {dt2.get_depth()}")
print(f"  Leaf nodes   : {dt2.get_n_leaves()}")


DT-2 (Pruned, max_depth=4) trained successfully.
  Tree depth   : 4
  Leaf nodes   : 16


#task 2c - Visualize DT-1(Fully Grown)

In [ ]:
# Visualise DT-1 using matplotlib plot_tree
# Limit display depth to 3 for readability (tree is very large)
fig1, ax1 = plt.subplots(figsize=(24, 8))
plot_tree(
    dt1,
    max_depth=3,                          # show top 3 levels for readability
    feature_names=X_reg.columns.tolist(),
    filled=True,
    rounded=True,
    fontsize=7,
    ax=ax1
)
ax1.set_title('DT-1: Fully Grown Decision Tree Regressor (top 3 levels shown)', fontsize=14)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/CW_ML_20240856/dt1_fully_grown.png', dpi=150, bbox_inches='tight')
plt.show()
print("DT-1 visualisation saved as 'dt1_fully_grown.png'")


DT-1 visualisation saved as 'dt1_fully_grown.png'


#Task 2c-Visualize DT-2(pruned, max_depth=4)

In [ ]:
# Visualise DT-2 – pruned tree is compact enough to display fully
fig2, ax2 = plt.subplots(figsize=(22, 10))
plot_tree(
    dt2,
    feature_names=X_reg.columns.tolist(),
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax2
)
ax2.set_title('DT-2: Pruned Decision Tree Regressor (max_depth=4)', fontsize=14)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/CW_ML_20240856/dt2_pruned.png', dpi=150, bbox_inches='tight')
plt.show()
print("DT-2 visualisation saved as 'dt2_pruned.png'")


DT-2 visualisation saved as 'dt2_pruned.png'


#task 3a -Evaluate DT-1 and DT-2 (MSE, MAE, R^2)

In [ ]:
# Generate predictions on test set using DT-1 (fully grown)
dt1_pred = dt1.predict(X_test_r)

# Generate predictions on test set using DT-2 (pruned)
dt2_pred = dt2.predict(X_test_r)

# Define a function to print all three regression metrics clearly
def reg_metrics(name, y_true, y_pred):
    # Calculate Mean Squared Error
    mse = mean_squared_error(y_true, y_pred)
    # Calculate Mean Absolute Error
    mae = mean_absolute_error(y_true, y_pred)
    # Calculate R-Squared score
    r2  = r2_score(y_true, y_pred)
    # Print results
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  MSE       : {mse:,.2f}")
    print(f"  RMSE      : {mse**0.5:,.2f}")
    print(f"  MAE       : {mae:,.2f}")
    print(f"  R-Squared : {r2:.4f}")
    # Return as dictionary for summary table
    return {'Model': name, 'MSE': mse, 'RMSE': mse**0.5, 'MAE': mae, 'R²': r2}

# Evaluate DT-1 fully grown model
results_dt1 = reg_metrics('DT-1 (Fully Grown)', y_test_r, dt1_pred)

# Evaluate DT-2 pruned model
results_dt2 = reg_metrics('DT-2 (Pruned, max_depth=4)', y_test_r, dt2_pred)

# Build and display regression metrics summary table
reg_summary = pd.DataFrame([results_dt1, results_dt2]).set_index('Model')
print("\nRegression Metrics Summary:")
print(reg_summary.round(2).to_string())



  DT-1 (Fully Grown)
  MSE       : 8,464,656.20
  RMSE      : 2,909.41
  MAE       : 992.28
  R-Squared : 0.9946

  DT-2 (Pruned, max_depth=4)
  MSE       : 162,871,123.97
  RMSE      : 12,762.10
  MAE       : 8,549.24
  R-Squared : 0.8956

Regression Metrics Summary:
                                     MSE      RMSE      MAE    R²
Model                                                            
DT-1 (Fully Grown)          8.464656e+06   2909.41   992.28  0.99
DT-2 (Pruned, max_depth=4)  1.628711e+08  12762.10  8549.24  0.90


#Actual vs Predicted Plot - DT-1 and DT-2

In [ ]:
# Plot actual vs predicted values for both regression models
fig_avp = go.Figure()

# Add DT-1 scatter
fig_avp.add_trace(go.Scatter(
    x=y_test_r, y=dt1_pred,
    mode='markers',
    name='DT-1 (Fully Grown)',
    marker=dict(color='#636EFA', opacity=0.4, size=4)
))

# Add DT-2 scatter
fig_avp.add_trace(go.Scatter(
    x=y_test_r, y=dt2_pred,
    mode='markers',
    name='DT-2 (Pruned)',
    marker=dict(color='#EF553B', opacity=0.4, size=4)
))

# Add perfect prediction line
min_val = int(y_test_r.min())
max_val = int(y_test_r.max())
fig_avp.add_trace(go.Scatter(
    x=[min_val, max_val], y=[min_val, max_val],
    mode='lines', name='Perfect Prediction',
    line=dict(color='green', dash='dash', width=2)
))

fig_avp.update_layout(
    title='Actual vs Predicted – Maximum Loan Amount',
    xaxis_title='Actual Maximum Loan Amount (£)',
    yaxis_title='Predicted Maximum Loan Amount (£)',
    title_x=0.5, width=700, height=480
)
fig_avp.show()


# Actual vs Predicted Plot - DT - 1 and DT-2

In [ ]:
# Plot actual vs predicted values for both DT models on one scatter chart
fig_avp = go.Figure()

# Add DT-1 scatter points
fig_avp.add_trace(go.Scatter(
    x=y_test_r, y=dt1_pred,
    mode='markers',
    name='DT-1 (Fully Grown)',
    marker=dict(color='#636EFA', opacity=0.4, size=4)
))

# Add DT-2 scatter points
fig_avp.add_trace(go.Scatter(
    x=y_test_r, y=dt2_pred,
    mode='markers',
    name='DT-2 (Pruned)',
    marker=dict(color='#EF553B', opacity=0.4, size=4)
))

# Add perfect prediction diagonal reference line
min_val = int(y_test_r.min())
max_val = int(y_test_r.max())
fig_avp.add_trace(go.Scatter(
    x=[min_val, max_val], y=[min_val, max_val],
    mode='lines', name='Perfect Prediction',
    line=dict(color='green', dash='dash', width=2)
))

# Update chart layout
fig_avp.update_layout(
    title='Actual vs Predicted – Maximum Loan Amount (DT-1 vs DT-2)',
    xaxis_title='Actual Maximum Loan Amount (£)',
    yaxis_title='Predicted Maximum Loan Amount (£)',
    title_x=0.5, width=700, height=480
)
fig_avp.show()


#Feature Importance -DT-2(Pruned)

In [ ]:
# Extract feature importances from the pruned DT-2 model
importance_df = pd.DataFrame({
    'Feature'   : X_reg.columns,
    'Importance': dt2.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot top 10 most important features as a horizontal bar chart
fig_imp = px.bar(
    importance_df.head(10),
    x='Importance', y='Feature',
    orientation='h',
    title='DT-2 – Top 10 Feature Importances',
    color='Importance',
    color_continuous_scale='Blues',
    labels={'Importance': 'Importance Score', 'Feature': 'Feature Name'}
)
# Reverse y-axis so most important feature is at the top
fig_imp.update_layout(yaxis=dict(autorange='reversed'), title_x=0.5)
fig_imp.show()

# Print top 5 features for reference
print("Top 5 Features by Importance (DT-2):")
print(importance_df.head(5).to_string(index=False))


Top 5 Features by Importance (DT-2):
           Feature  Importance
            income    0.912613
               age    0.087387
 employment_length    0.000000
       loan_amount    0.000000
loan_interest_rate    0.000000


#Task 3d - Predict Maximum Loan Amount for client 60256

In [ ]:
# Build Client 60256 feature row matching the regression dataset columns exactly
# Encode categorical values the same way as Notebook 1 preprocessing:
# payment_default_on_file: No = 0, Yes = 1 (Label Encoding)
# home_ownership and loan_intent: one-hot encoded with drop_first=True

# Create client record aligned to regression feature columns
client_60256 = pd.DataFrame(columns=X_reg.columns)

# Add a single row of zeros as starting point
client_60256.loc[0] = 0

# Fill in numerical feature values for Client 60256
client_60256.loc[0, 'age']                   = 56
client_60256.loc[0, 'income']                = 57000
client_60256.loc[0, 'employment_length']     = 15
client_60256.loc[0, 'loan_amount']           = 25700
client_60256.loc[0, 'loan_interest_rate']    = 23.0
client_60256.loc[0, 'loan_income_ratio']     = 0.10
client_60256.loc[0, 'payment_default_on_file'] = 0   # No = 0
client_60256.loc[0, 'credit_history_length'] = 35

# Set one-hot encoded columns for home_ownership = RENT
# (drop_first=True in NB1 dropped the first alphabetical category)
# Check which home_ownership column exists in dataset and set accordingly
for col in X_reg.columns:
    if 'home_ownership' in col and 'RENT' in col.upper():
        client_60256.loc[0, col] = 1

# Set one-hot encoded columns for loan_intent = MEDICAL
for col in X_reg.columns:
    if 'loan_intent' in col and 'MEDICAL' in col.upper():
        client_60256.loc[0, col] = 1

# Ensure correct data types matching training data
client_60256 = client_60256.astype(float)

# Predict maximum loan amount using DT-2 (pruned model — best model)
prediction = dt2.predict(client_60256)[0]

# Print prediction result
print("="*55)
print("  Client 60256 – Maximum Loan Amount Prediction")
print("="*55)
print(f"  Model Used         : DT-2 (Pruned, max_depth=4)")
print(f"  Predicted Max Loan : £{prediction:,.2f}")
print("="*55)


  Client 60256 – Maximum Loan Amount Prediction
  Model Used         : DT-2 (Pruned, max_depth=4)
  Predicted Max Loan : £85,093.65


/tmp/ipykernel_21994/1366436709.py:18: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.1' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.



In [ ]:
# Show the decision path taken through DT-2 for Client 60256
# This explains step by step how the prediction was reached
from sklearn.tree import _tree

# Get the tree structure
tree_ = dt2.tree_
feature_names = X_reg.columns.tolist()

# Define recursive function to trace decision path
def trace_path(node, client_row, depth=0):
    indent = "  " * depth
    # Check if this is a leaf node
    if tree_.children_left[node] == _tree.TREE_LEAF:
        # Print the leaf node prediction value
        print(f"{indent}→ LEAF: Predicted value = £{tree_.value[node][0][0]:,.2f}")
        return
    # Get feature name and threshold for this split node
    feature = feature_names[tree_.feature[node]]
    threshold = tree_.threshold[node]
    # Get client value for this feature
    client_val = float(client_row[feature].values[0])
    # Decide which branch to follow
    if client_val <= threshold:
        print(f"{indent}Node: {feature} = {client_val} <= {threshold:.4f} → go LEFT")
        trace_path(tree_.children_left[node], client_row, depth + 1)
    else:
        print(f"{indent}Node: {feature} = {client_val} > {threshold:.4f} → go RIGHT")
        trace_path(tree_.children_right[node], client_row, depth + 1)

# Trace the full decision path for Client 60256 through DT-2
print("DT-2 Decision Path for Client 60256:")
print("-"*55)
trace_path(0, client_60256)


DT-2 Decision Path for Client 60256:
-------------------------------------------------------
Node: income = 57000.0 <= 78756.0000 → go LEFT
  Node: income = 57000.0 > 53022.0000 → go RIGHT
    Node: age = 56.0 > 28.5000 → go RIGHT
      Node: income = 57000.0 <= 64999.5000 → go LEFT
        → LEAF: Predicted value = £85,093.65


#Final Summary

In [ ]:
# Print complete summary of all results from Notebook 3
print("="*60)
print("  PYTHON NOTEBOOK 3 – FINAL SUMMARY")
print("="*60)

# Part A summary
print("\nPART A – Ensemble Classification (Loan Approval Status)")
print("-"*60)
print(f"  Base Learner 1 : Naive Bayes (NB)")
print(f"  Base Learner 2 : Logistic Regression (LR)")
print(f"  Ensemble Type  : Soft Voting Classifier")
print(f"  NB  Recall     : {recall_score(y_test_c, nb_pred, zero_division=0):.4f}")
print(f"  LR  Recall     : {recall_score(y_test_c, lr_pred, zero_division=0):.4f}")
print(f"  Ensemble Recall: {vc_rec:.4f}")
print(f"  Ensemble AUC   : {vc_auc:.4f}")

# Part B summary
print("\nPART B – Regression Decision Trees (Maximum Loan Amount)")
print("-"*60)
print(f"  DT-1 (Fully Grown) MAE : £{mean_absolute_error(y_test_r, dt1_pred):,.2f}")
print(f"  DT-2 (Pruned)      MAE : £{mean_absolute_error(y_test_r, dt2_pred):,.2f}")
print(f"  DT-1 R-Squared         : {r2_score(y_test_r, dt1_pred):.4f}")
print(f"  DT-2 R-Squared         : {r2_score(y_test_r, dt2_pred):.4f}")
print(f"  Client 60256 Predicted : £{prediction:,.2f} (DT-2)")

print("\n" + "="*60)
print("Notebook 3 complete. All results ready for the Analysis Report.")
print("="*60)


  PYTHON NOTEBOOK 3 – FINAL SUMMARY

PART A – Ensemble Classification (Loan Approval Status)
------------------------------------------------------------
  Base Learner 1 : Naive Bayes (NB)
  Base Learner 2 : Logistic Regression (LR)
  Ensemble Type  : Soft Voting Classifier
  NB  Recall     : 0.2834
  LR  Recall     : 0.4248
  Ensemble Recall: 0.3577
  Ensemble AUC   : 0.8775

PART B – Regression Decision Trees (Maximum Loan Amount)
------------------------------------------------------------
  DT-1 (Fully Grown) MAE : £992.28
  DT-2 (Pruned)      MAE : £8,549.24
  DT-1 R-Squared         : 0.9946
  DT-2 R-Squared         : 0.8956
  Client 60256 Predicted : £85,093.65 (DT-2)

Notebook 3 complete. All results ready for the Analysis Report.
